# cudf-bench: first GPU benchmarks (Phases 0–1)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

This notebook: (1) verifies the GPU, (2) gets cuDF, (3) clones the repo, (4) runs the same benchmark grid under pandas and cuDF, (5) shows the speedups.

In [ ]:
# Phase 0 check: a T4 should be listed below. If not, fix the runtime type first.
!nvidia-smi

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd cudf-bench
%pip install -q -e .

## Run the benchmarks

Same ops, same deterministic 10M-row data, two backends. The pandas run takes a few minutes — that is the point.

In [ ]:
!python -m benchmark.runner --backend pandas --rows 1e7 --out results/results.csv

In [ ]:
!python -m benchmark.runner --backend cudf --rows 1e7 --out results/results.csv

In [ ]:
from benchmark.report import load_results, speedup_table, plot_speedups

results = load_results("results/results.csv")
speedup_table(results)

In [ ]:
plot_speedups(results, backend="cudf", baseline="pandas");

## Sanity checklist (Phase 2 gate)

- cuDF should win by roughly 10–50x on `groupby_agg` / `inner_join` at 10M rows. If it does not, the stopwatch is broken — debug the harness before trusting anything else.
- Rerun with `--rows 1e6,1e7,3e7` and check time grows with size for both backends.
- Try `--skew 0,1.1` and watch what happens to the join.

## Save your results

Download `results/results.csv` (Files sidebar) and commit it to the repo, or push directly:
```
!git config user.email you@example.com && git config user.name yourname
!git add results/ && git commit -m "Add T4 benchmark results" && git push
```
(pushing from Colab needs a GitHub token — downloading the CSV and committing locally is simpler)